In [1]:
from pathlib import Path
from typing import List, Tuple, Dict
import json, random, itertools, statistics as st

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import KFold, train_test_split
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import BallTree

import torch
from transformers import (
    AutoTokenizer,
    DataCollatorForTokenClassification,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
)

from seqeval.metrics import f1_score, classification_report
from tqdm.auto import tqdm

c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Configuração e Verificação Inicial

In [2]:
JSON_PATH = "data/geocorpus-v2.json"        # ajuste se estiver noutra pasta
SEED_GLOBAL = 42
random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)

# ---------- ler o arquivo ----------
with open(JSON_PATH, encoding="utf-8") as f:
    raw = json.load(f)

# cada entrada já tem tokens + ner_tokens

MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

In [3]:
raw

[{'sentences': 'na escala de tempo geológico , o lopingiano é a época do período permiano da era paleozoica do éon fanerozoico que está compreendida entre 260 milhões e 400 mil anos e 251 milhões de anos atrás , aproximadamente .',
  'tokens': ['na',
   'escala',
   'de',
   'tempo',
   'geológico',
   ',',
   'o',
   'lopingiano',
   'é',
   'a',
   'época',
   'do',
   'período',
   'permiano',
   'da',
   'era',
   'paleozoica',
   'do',
   'éon',
   'fanerozoico',
   'que',
   'está',
   'compreendida',
   'entre',
   '260',
   'milhões',
   'e',
   '400',
   'mil',
   'anos',
   'e',
   '251',
   'milhões',
   'de',
   'anos',
   'atrás',
   ',',
   'aproximadamente',
   '.'],
  'ner_tokens': ['O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'B-epoca',
   'O',
   'O',
   'O',
   'O',
   'O',
   'B-periodo',
   'O',
   'O',
   'B-era',
   'O',
   'O',
   'B-eon',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
 

In [4]:
records_geo = [
    {
        "sentence_id": i,
        "tokens"     : item["tokens"],
        "ner_tags"   : item["ner_tokens"],
    }
    for i, item in enumerate(raw)
]

geocorpus_full = Dataset.from_list(records_geo)

In [5]:
pd.DataFrame(geocorpus_full[:5])

,sentence_id,tokens,ner_tags
0,0,"[na, escala, de, tempo, geológico, ,, o, lopin...","[O, O, O, O, O, O, O, B-epoca, O, O, O, O, O, ..."
1,1,"[o, lopingiano, constitui, a, subdivisão, post...","[O, B-epoca, O, O, O, O, O, B-periodo, O, O]"
2,2,"[ao, término, do, lopingiano, houve, um, perío...","[O, O, O, B-epoca, O, O, O, O, O, O, O, O, O, ..."
3,3,"[os, dois, andares, do, lopingiano, devem, o, ...","[O, O, O, O, B-epoca, O, O, O, O, O, O, O, O, ..."
4,4,"[como, parte, da, ultima, revisão, da, estrtig...","[O, O, O, O, O, O, O, O, B-periodo, O, O, O, B..."


In [6]:
# lista de rótulos (ordem alfabética garante consistência entre runs)
label_list = sorted({l for sent in geocorpus_full["ner_tags"] for l in sent})
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
NUM_LABELS = len(label_list)

In [7]:
id2label

{0: 'B-ambienteSedimentacao',
 1: 'B-baciaSedimentar',
 2: 'B-bentonico',
 3: 'B-campoPetrolifero',
 4: 'B-constituinteRochaSedimentar',
 5: 'B-contextoGeologicoDeBacia',
 6: 'B-elementoQuimico',
 7: 'B-eon',
 8: 'B-epoca',
 9: 'B-era',
 10: 'B-estratigrafia',
 11: 'B-estruturaGeologica',
 12: 'B-estruturaSedimentar',
 13: 'B-fosseis',
 14: 'B-geomorfologia',
 15: 'B-granulometria',
 16: 'B-idade',
 17: 'B-magmaticas',
 18: 'B-metamorficas',
 19: 'B-mineral',
 20: 'B-periodo',
 21: 'B-planctonico',
 22: 'B-procedimentoMetodologico',
 23: 'B-sedimentaresCarbonaticas',
 24: 'B-sedimentaresOrganicas',
 25: 'B-sedimentaresQuimicas',
 26: 'B-sedimentaresSiliciclasticas',
 27: 'B-sistemaPetrolifero',
 28: 'B-unidadeEstratigrafica',
 29: 'B-unidadeGeotectonica',
 30: 'I-ambienteSedimentacao',
 31: 'I-baciaSedimentar',
 32: 'I-bentonico',
 33: 'I-campoPetrolifero',
 34: 'I-constituinteRochaSedimentar',
 35: 'I-contextoGeologicoDeBacia',
 36: 'I-epoca',
 37: 'I-era',
 38: 'I-estratigrafia',
 39

In [8]:
NUM_LABELS

57

# Splits

In [9]:
# --- hold-out 80/20 (standard para este corpus) -----------------
standard_geo = geocorpus_full.train_test_split(
    test_size=0.2, seed=SEED_GLOBAL
)

# --- 30 divisões aleatórias -------------------------------------
def random_splits(
    ds: Dataset, test_size=0.2, seeds: List[int] = range(10)
) -> List[DatasetDict]:
    triples = []
    for s in seeds:
        train, dev = ds.train_test_split(test_size=test_size, seed=s).values()
        triples.append(DatasetDict(train=train, dev=dev))
    return triples

# --- heurística comprimento (20 % piores casos) -----------------
def split_heur_length(ds, top_pct=0.20):
    lengths   = np.array([len(t) for t in ds["tokens"]])
    thr       = np.percentile(lengths, 100*(1-top_pct))
    idx_long  = np.where(lengths >= thr)[0]
    idx_short = np.where(lengths <  thr)[0]
    return DatasetDict(
        train=ds.select(idx_short.tolist()),
        dev  =ds.select(idx_long.tolist())
    )

# --- heurística raridade ----------------------------------------
def split_heur_rare(ds, freq_thr=5):
    freq = {}
    for sent in ds["tokens"]:
        for w in sent:
            freq[w.lower()] = freq.get(w.lower(), 0) + 1
    rare = {w for w,c in freq.items() if c <= freq_thr}
    keep_dev = [any(w.lower() in rare for w in sent) for sent in ds["tokens"]]
    idx_dev   = [i for i,b in enumerate(keep_dev) if b]
    idx_train = [i for i,b in enumerate(keep_dev) if not b]
    return DatasetDict(train=ds.select(idx_train), dev=ds.select(idx_dev))

# --- adversarial (versão rápida, 10 % do corpus) ----------------
def split_adversarial(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embeds = model.encode([" ".join(t) for t in ds["tokens"]], show_progress_bar=False)
    tree = BallTree(embeds, leaf_size=40)

    idx_train, idx_test = set(range(len(ds))), []
    # semente = ponto mais central
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    idx_train.remove(seed_idx)
    idx_test.append(seed_idx)
    print(k)
    while len(idx_test) < k:
        print(len(idx_test))
        dists, _ = tree.query(embeds[list(idx_train)], k=1, return_distance=True)
        nxt = list(idx_train)[int(np.argmax(dists))]
        idx_train.remove(nxt)
        idx_test.append(nxt)

    return DatasetDict(
        train=ds.select(sorted(idx_train)),
        dev=ds.select(sorted(idx_test)),
    )

    # Maximizando Wassertein Distance


def split_adversarial_fast(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    """
    Farthest-Point Sampling aproximando Wasserstein – versão vetorizada.
    Seleciona pct_test (~20 %) das sentenças como conjunto 'dev'.
    """
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    embeds = model.encode(
        [" ".join(t) for t in ds["tokens"]],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # acelera distância euclidiana ≈ cos
    )

    n = embeds.shape[0]
    idx_all = np.arange(n)

    # 1) ponto mais "central" (norma mais distante da média)
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    selected = [seed_idx]

    # 2) vetor de distâncias mínimas a qualquer ponto já escolhido
    min_dists = np.linalg.norm(embeds - embeds[seed_idx], axis=1)

    while len(selected) < k:
        next_idx = np.argmax(min_dists)
        selected.append(next_idx)

        # atualiza min_dists com a distância ao novo ponto — tudo de uma vez
        d_new = np.linalg.norm(embeds - embeds[next_idx], axis=1)
        min_dists = np.minimum(min_dists, d_new)

    train_idx = np.setdiff1d(idx_all, selected, assume_unique=True)

    return DatasetDict(
        train=ds.select(train_idx.tolist()),
        dev=ds.select(selected),
    )

In [10]:
standard_geo

DatasetDict({
    train: Dataset({
        features: ['sentence_id', 'tokens', 'ner_tags'],
        num_rows: 4217
    })
    test: Dataset({
        features: ['sentence_id', 'tokens', 'ner_tags'],
        num_rows: 1055
    })
})

In [11]:
standard_geo["train"]

Dataset({
    features: ['sentence_id', 'tokens', 'ner_tags'],
    num_rows: 4217
})

In [12]:
standard_geo = DatasetDict(train=standard_geo["train"], dev=standard_geo["test"])
random_geo = random_splits(geocorpus_full)
heur_len_geo = split_heur_length(geocorpus_full)
heur_rare_geo = split_heur_rare(geocorpus_full)
advers_geo = split_adversarial_fast(geocorpus_full)

Batches: 100%|██████████| 165/165 [00:57<00:00,  2.88it/s]


# Tokenização e Métricas

In [13]:
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME, model_max_length=512)
collator   = DataCollatorForTokenClassification(tokenizer, padding=True)

In [14]:
def tokenize(batch):
    enc = tokenizer(batch["tokens"],
                    is_split_into_words=True,
                    truncation=True,
                    padding=True)           # <- padding alinhado
    labels = []
    for i in range(len(batch["tokens"])):
        word_ids   = enc.word_ids(batch_index=i)
        sent_labels = batch["ner_tags"][i]
        aligned = []
        last = None
        for wid in word_ids:
            if wid is None:
                aligned.append(-100)
            elif wid != last:
                aligned.append(label2id[sent_labels[wid]])
                last = wid
            else:
                aligned.append(-100)
        labels.append(aligned)
    enc["labels"] = labels
    return enc

In [15]:
def compute_metrics(p):
    logits, labels = p
    preds = np.argmax(logits, -1)
    out_pred, out_true = [], []
    for p_i, l_i in zip(preds, labels):
        mask = l_i != -100
        out_pred.append([id2label[idx] for idx in p_i[mask]])
        out_true.append([id2label[idx] for idx in l_i[mask]])
    return {"f1": f1_score(out_true, out_pred)}

In [16]:
def run_experiment_geo(split: DatasetDict, seed: int):
    encoded = split.map(tokenize, batched=True,
                        remove_columns=split["train"].column_names)
    model   = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id,
        torch_dtype="auto"
    )
    args = TrainingArguments(
        output_dir=f"tmp/geo_seed{seed}",
        eval_strategy="no",
        learning_rate=2e-5,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=3,
        fp16=True,
        seed=seed,
        logging_strategy="steps",
        logging_steps=50,
        save_strategy="no",
        report_to="none",
    )
    trainer = Trainer(model=model,
                      args=args,
                      data_collator=collator,
                      train_dataset=encoded["train"],
                      compute_metrics=compute_metrics)
    trainer.train()
    return trainer, trainer.evaluate(encoded["dev"])["eval_f1"], encoded

In [18]:
rand_geo_scores

NameError: name 'rand_geo_scores' is not defined

In [17]:
# Standard
# std_geo = run_experiment_geo(standard_geo, SEED_GLOBAL)

# 30 random
rand_geo_scores = [
    run_experiment_geo(split, i) for i, split in enumerate(random_geo)
]



# # Heuristics + adversarial
# f1_len_geo  = run_experiment_geo(heur_len_geo, 111)
# trainer, f1_rare_geo, encoded = run_experiment_geo(heur_rare_geo, 222)
# f1_adv_geo  = run_experiment_geo(advers_geo,    333)


Map:   0%|          | 0/4217 [00:00<?, ? examples/s]

Map: 100%|██████████| 1055/1055 [00:00<00:00, 2415.83 examples/s]
Some weights of BertForTokenClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
50,1.049800
100,0.430400
150,0.330000
200,0.277700
250,0.235600
300,0.208800
350,0.182400
400,0.185900
450,0.163800
500,0.162300


Map: 100%|██████████| 1055/1055 [00:00<00:00, 3575.75 examples/s]
Some weights of BertForTokenClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
50,1.011700
100,0.374600
150,0.321900
200,0.262600
250,0.239400
300,0.223800
350,0.207500
400,0.178000
450,0.150800
500,0.133600


Map: 100%|██████████| 1055/1055 [00:00<00:00, 3088.81 examples/s]
Some weights of BertForTokenClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
50,1.056500
100,0.396100
150,0.337700
200,0.295200
250,0.260400
300,0.223500


KeyboardInterrupt: 

In [ ]:



# print(f"Geo - standard F1  : {std_geo:.3f}")
print(f"Geo - random  F1   : {st.mean(rand_geo_scores):.3f} ± {st.stdev(rand_geo_scores):.3f}")
# print(f"Geo - heur-len F1  : {f1_len_geo:.3f}")



# print(f"Geo - heur-rare F1 : {f1_rare_geo:.3f}")
# print(f"Geo - adversarial  : {f1_adv_geo:.3f}")

Geo - heur-rare F1 : 0.000
